# Concise computational results
One matched-case table and two publication figures. Change the settings, then **Run All**.

The table focuses on certification at $n=20$. The performance profile covers $n\ge20$;
the boxplot shows the remaining gaps for $n\ge30$. Defaults are explicit choices,
not automatic selection of favorable results. All methods use the same cases in each output.
The workbook is read only; exports are saved separately.

In [ ]:
from pathlib import Path
import hashlib, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

ROOT = Path.cwd()
if not (ROOT / "all_v1.xlsx").exists():
    ROOT = ROOT.parent
WORKBOOK = ROOT / "all_v1.xlsx"  # edit if needed
SHEET = "runs_large"             # falls back to all_v1 when renamed
METHODS = ["compact", "PW", "LR-D", "LR-A", "LR-SA"]
TAUS = [1.5, 2.0, 3.0]           # use e.g. [2.0] for a single norm
TABLE_SIZES = [20]
PROFILE_SIZES = [20, 30, 40, 50]
BOX_SIZES = [30, 40, 50]
TIME_LIMIT = 3600.0
REL_TOL, ABS_TOL = 1e-4, 1e-6
OUT = ROOT / "results" / "paper_summary_v1"
OUT.mkdir(parents=True, exist_ok=True)
COLORS = dict(zip(METHODS, ["#555555", "#0072B2", "#D55E00", "#009E73", "#CC79A7"]))
plt.rcParams.update({"font.size": 10, "axes.spines.top": False,
                     "axes.spines.right": False, "pdf.fonttype": 42,
                     "savefig.dpi": 300})

### Read and match cases
A case is `(instance, p, tau)`. Missing runs are **not** treated as timeouts:
only cases present for every selected method are compared. An available run is certified
when its finite bounds close the common tolerance. Solver status is retained for auditing.
Conflicting duplicates, mixed projector engines, and invalid bounds stop execution.

In [ ]:
book = pd.ExcelFile(WORKBOOK)
used_sheet = SHEET if SHEET in book.sheet_names else ("all_v1" if "all_v1" in book.sheet_names else None)
if used_sheet is None:
    raise ValueError(f"Choose a results sheet from {book.sheet_names}")
raw = pd.read_excel(book, sheet_name=used_sheet)
KEY = ["instance", "p", "tau"]
required = KEY + ["n", "configuration", "objective", "lower_bound", "runtime", "status"]
missing = set(required) - set(raw.columns)
if missing:
    raise ValueError(f"Missing columns: {sorted(missing)}")
df = raw.loc[raw.configuration.isin(METHODS)].copy()
for col in ["n", "p", "tau", "objective", "lower_bound", "runtime"]:
    df[col] = pd.to_numeric(df[col], errors="coerce")
df = df.loc[df.tau.isin(TAUS)].copy()
if df.empty:
    raise ValueError("No rows match the selected norms and methods")
if df[KEY + ["n"]].isna().any().any():
    raise ValueError("Missing case identifiers")
if df.duplicated(KEY + ["configuration"]).any():
    raise ValueError("Duplicate case/method rows: separate campaigns before comparing")
if "engine" in df:
    engines = df.loc[df.configuration.ne("compact"), "engine"].dropna().unique()
    if len(engines) > 1:
        raise ValueError(f"Mixed projector engines: {engines}")
if (~np.isfinite(df.runtime) | df.runtime.le(0)).any():
    raise ValueError("Runtime must be positive and finite")
finite = np.isfinite(df.objective) & np.isfinite(df.lower_bound)
if (finite & (df.lower_bound > df.objective + 1e-5 * np.maximum(1, df.objective.abs()))).any():
    raise ValueError("A lower bound exceeds its objective: inspect source rows")
difference = (df.objective - df.lower_bound).clip(lower=0)
df["certified"] = finite & (difference <= np.maximum(ABS_TOL, REL_TOL * df.objective.abs()) + 1e-8)
df["gap_pct"] = np.where(finite, 100 * difference / np.maximum(df.objective.abs(), 1e-12), np.nan)
# PAR2 penalizes an unsuccessful run by twice the nominal limit; it is a score,
# not the observed runtime. No faster failure is rewarded.
df["par2"] = np.where(df.certified, df.runtime, 2 * TIME_LIMIT)
coverage = df.groupby(KEY).configuration.nunique()
complete_keys = coverage[coverage.eq(len(METHODS))].index
matched = df.set_index(KEY).loc[lambda x: x.index.isin(complete_keys)].reset_index()
if matched.empty:
    raise ValueError("No cases have all selected methods; choose fewer methods or await more results")
assert matched.groupby(KEY).size().eq(len(METHODS)).all()
for col in ["n"]:
    if df.groupby(KEY)[col].nunique().gt(1).any():
        raise ValueError(f"Inconsistent {col} within a case")
print(f"Sheet: {used_sheet}; rows selected: {len(df)}; complete matched cases: {len(complete_keys)}; incomplete cases excluded: {(coverage < len(METHODS)).sum()}")
coverage_table = matched.drop_duplicates(KEY).groupby(["tau", "n"]).size().rename("matched cases").to_frame()
display(coverage_table)
matched.to_csv(OUT / "matched_runs.csv", index=False)

def population(sizes):
    selected = matched.loc[matched.n.isin(sizes)].copy()
    if selected.empty:
        raise ValueError(f"No matched cases at sizes {sizes}; change settings")
    return selected

def save_plot(fig, stem):
    for extension in ("pdf", "png"):
        fig.savefig(OUT / f"{stem}.{extension}", bbox_inches="tight")

### One short table: certification and cost
**Solved/N**, **PAR2 score (seconds)**, and **mean final relative gap (%)**.
Failures receive `2 × TIME_LIMIT` in PAR2. Bound-based gaps are recomputed as
$(U-L)/|U|$, rather than using the source's `max(1, |U|)` normalization.
Missing gaps are counted and shown in the caption; they are not replaced by zero.

In [ ]:
tab = population(TABLE_SIZES)
summary = tab.groupby("configuration").agg(N=("runtime", "size"), Solved=("certified", "sum"),
    PAR2_s=("par2", "mean"), Gap_pct=("gap_pct", "mean"), Gap_N=("gap_pct", "count")).reindex(METHODS)
display(summary.round(2))
N = int(summary.N.iloc[0])
lines = [r"\begin{table}[t]", r"\centering", r"\begin{tabular}{lrrr}", r"\hline",
         r"Method & Solved/$N$ & PAR2 (s) & Gap (\%) \\", r"\hline"]
for method, row in summary.iterrows():
    gap = "--" if pd.isna(row.Gap_pct) else f"{row.Gap_pct:.2f}"
    lines.append(f"{method} & {int(row.Solved)}/{int(row.N)} & {row.PAR2_s:.1f} & {gap} " + r"\\")
lines += [r"\hline", r"\end{tabular}"]
caption = (f"Partial results on {N} matched cases with $n\in\{{{','.join(map(str,TABLE_SIZES))}\}}$ "
           f"and $\tau\in\{{{','.join(f'{t:g}' for t in TAUS)}\}}$. "
           "Solved denotes certification at a common tolerance. PAR2 assigns twice the nominal "
           f"{TIME_LIMIT:g}-second limit to unsuccessful runs. Gaps are mean final relative gaps.")
if summary.Gap_N.lt(summary.N).any():
    caption += " Missing gaps are excluded from the means; see the accompanying CSV for counts."
lines += [r"\caption{" + caption + "}", r"\label{tab:projector-summary}", r"\end{table}"]
(OUT / "summary.tex").write_text("\n".join(lines) + "\n")
summary.to_csv(OUT / "summary.csv")
print("LaTeX: ", OUT / "summary.tex")

### Performance profile: speed and reliability
For case $i$, $r_{i,m}=t_{i,m}/\min_s t_{i,s}$ among certified solves.
An unsuccessful solve has ratio $\infty$. Cases unsolved by every method remain
in the denominator: plateaus therefore measure the certified fraction of **all**
matched cases, not just easy cases. Higher curves are better. Times include each
method's auxiliary SDP work; the shared warm start is excluded, as in the source.

In [ ]:
def performance_ratios(times, success):
    cost = np.where(success, times, np.inf)
    best = np.min(cost, axis=1)
    ratios = np.full(cost.shape, np.inf)
    valid = np.isfinite(best)
    ratios[valid] = cost[valid] / best[valid, None]
    return ratios

# Check failure handling, including a case unsolved by everybody.
check = performance_ratios(np.array([[1., 2.], [1., 1.]]), np.array([[True, False], [False, False]]))
assert check[0, 0] == 1 and np.isinf(check[0, 1]) and np.isinf(check[1]).all()
prof = population(PROFILE_SIZES)
times = prof.pivot(index=KEY, columns="configuration", values="runtime").reindex(columns=METHODS)
success = prof.pivot(index=KEY, columns="configuration", values="certified").reindex(columns=METHODS)
ratios = performance_ratios(times.to_numpy(), success.to_numpy(dtype=bool))
finite_ratios = ratios[np.isfinite(ratios)]
xmax = max(2., 2. ** np.ceil(np.log2(finite_ratios.max()))) if len(finite_ratios) else 2.
fig, ax = plt.subplots(figsize=(6.2, 3.6))
for j, method in enumerate(METHODS):
    x = np.unique(np.r_[1., ratios[np.isfinite(ratios[:, j]), j], xmax])
    y = (ratios[:, j, None] <= x[None, :]).mean(axis=0)
    ax.step(x, y, where="post", color=COLORS[method], lw=1.8,
            linestyle=["--", "-", "-.", ":", "-"][j],
            label=f"{method} ({int(success[method].sum())}/{len(times)})")
ax.set(xscale="log", xlim=(1, xmax), ylim=(0, 1.03),
       xlabel="Time relative to the fastest certified method",
       ylabel="Fraction of matched cases certified",
       title=f"Performance profile: n in {PROFILE_SIZES} (N={len(times)})")
ax.grid(alpha=.2, which="both"); ax.legend(fontsize=8, loc="lower right")
fig.tight_layout(); save_plot(fig, "performance_profile"); plt.show()
profile_caption = (f"Performance profiles on {len(times)} matched cases. Higher curves indicate better "
                   "certification efficiency. Unsuccessful solves have infinite ratios; cases unsolved "
                   "by all methods remain in the denominator. Legend counts report certified cases.")

### Boxplot: quality of the remaining certificates
Lower gaps are better. Every method has the same cases, including solved ones
with near-zero gaps. Individual points are shown because the larger-instance
sample is still small. If any final gap is missing, that case is omitted for
**all** methods in this figure and the omission is reported.

In [ ]:
box = population(BOX_SIZES)
gaps = box.pivot(index=KEY, columns="configuration", values="gap_pct").reindex(columns=METHODS)
missing_gap_cases = int(gaps.isna().any(axis=1).sum())
gaps = gaps.dropna()
if gaps.empty:
    raise ValueError("No complete finite-gap cases for the boxplot")
fig, ax = plt.subplots(figsize=(6.2, 3.6))
artists = ax.boxplot([gaps[m].to_numpy() for m in METHODS], patch_artist=True,
                    widths=.55, showfliers=False, medianprops={"color": "black"})
rng = np.random.default_rng(0)
for j, method in enumerate(METHODS):
    artists["boxes"][j].set(facecolor=COLORS[method], alpha=.25)
    ax.scatter(j+1+rng.uniform(-.12, .12, len(gaps)), gaps[method],
               color=COLORS[method], s=22, alpha=.8, zorder=3)
ax.set_xticks(range(1, len(METHODS)+1), METHODS)
ax.set(ylabel="Final relative gap (%)", ylim=(0, None),
       title=f"Final certificates: n in {BOX_SIZES} (N={len(gaps)})")
ax.grid(axis="y", alpha=.2)
fig.tight_layout(); save_plot(fig, "gap_boxplot"); plt.show()
print(f"Cases omitted because at least one gap is missing: {missing_gap_cases}")
box_caption = (f"Final relative gaps on {len(gaps)} matched larger-instance cases. Lower is better. "
               "Boxes show quartiles and medians; points are individual cases. "
               f"{missing_gap_cases} cases with incomplete gaps are excluded for every method.")

### LaTeX snippets and a short factual takeaway
Use `\input{.../summary.tex}` for the table. Figures need `\usepackage{graphicx}`.
The snippets use filenames relative to the export folder; copy the PDFs and snippets
into Overleaf, or adjust the paths. The generated comparison is descriptive,
not a claim that the formulations alone cause the observed differences.

In [ ]:
figure_tex = []
for stem, caption_text, label in [("performance_profile", profile_caption, "fig:projector-profile"),
                                  ("gap_boxplot", box_caption, "fig:projector-gaps")]:
    figure_tex += [r"\begin{figure}[t]", r"\centering",
                   r"\includegraphics[width=0.85\linewidth]{"+stem+".pdf}",
                   r"\caption{"+caption_text+"}", r"\label{"+label+"}", r"\end{figure}", ""]
(OUT / "figures.tex").write_text("\n".join(figure_tex))
if "compact" in METHODS:
    compact = summary.loc["compact"]
    messages = [f"On {N} matched table cases, compact certifies {int(compact.Solved)}."]
    for method in METHODS:
        if method != "compact":
            row = summary.loc[method]
            messages.append(f"{method}: {int(row.Solved)}/{int(row.N)} certified; PAR2 {row.PAR2_s:.1f} s.")
    messages.append("These are partial results for the strengthened implementations.")
    takeaway = " ".join(messages)
    display(Markdown(takeaway))
    (OUT / "takeaway.txt").write_text(takeaway + "\n")
metadata = dict(workbook=str(WORKBOOK.resolve()), sheet=used_sheet,
                source_sha256=hashlib.sha256(WORKBOOK.read_bytes()).hexdigest(),
                methods=METHODS, taus=TAUS, table_sizes=TABLE_SIZES,
                profile_sizes=PROFILE_SIZES, box_sizes=BOX_SIZES,
                nominal_time_limit=TIME_LIMIT, relative_tolerance=REL_TOL,
                absolute_tolerance=ABS_TOL, matching="intersection of selected methods")
(OUT / "analysis_settings.json").write_text(json.dumps(metadata, indent=2))
print("Exports:", OUT)
print("summary.tex; performance_profile.pdf; gap_boxplot.pdf; figures.tex")